# Exploratory Data Analysis — Dropout Predict

Dataset: *Predict Students' Dropout and Academic Success* (UCI).

Notebook ini mengeksplorasi distribusi target, kualitas data, dan fitur yang paling berkaitan dengan status akademik mahasiswa.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

BASE = Path.cwd()
if not (BASE / 'data.csv').exists():
    BASE = BASE.parent
sys.path.insert(0, str(BASE / 'src'))
from config import TARGET
from data import load_data

plt.style.use('seaborn-v0_8-whitegrid')
df = load_data(BASE / 'data.csv')
df.shape

## 1. Distribusi target

Dataset memiliki tiga kelas dengan distribusi tidak seimbang. `Graduate` mendominasi, sedangkan `Enrolled` paling sedikit.

In [ ]:
counts = df[TARGET].value_counts()
print(counts)
print((counts / len(df) * 100).round(1))

ax = counts.plot(kind='bar', color=['#067647', '#d92d20', '#c97a08'], figsize=(7, 4))
ax.set_title('Distribusi status mahasiswa')
ax.set_xlabel('')
ax.set_ylabel('Jumlah mahasiswa')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 2. Kualitas data

Cek missing value dan duplikat.

In [ ]:
print('Missing value :', int(df.isna().sum().sum()))
print('Baris duplikat:', int(df.duplicated().sum()))
print('Jumlah fitur  :', df.shape[1] - 1)

df.describe().T[['mean', 'std', 'min', 'max']].round(2)

## 3. Rata-rata fitur per kelas

Bandingkan beberapa fitur numerik penting berdasarkan status akhir mahasiswa.

In [ ]:
important = [
    'Curricular units 1st sem (approved)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (grade)',
    'Admission grade',
    'Age at enrollment',
]
df.groupby(TARGET)[important].mean().round(2)

### Boxplot SKS lulus semester 2 vs status

In [ ]:
order = ['Dropout', 'Enrolled', 'Graduate']
data = [df.loc[df[TARGET] == label, 'Curricular units 2nd sem (approved)'] for label in order]

fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(data, tick_labels=order, patch_artist=True,
           boxprops=dict(facecolor='#e9edff'))
ax.set_title('SKS lulus semester 2 berdasarkan status')
ax.set_ylabel('SKS lulus')
plt.tight_layout()
plt.show()

## 4. Korelasi fitur numerik

Heatmap korelasi membantu melihat fitur yang redundan (mis. semester 1 vs semester 2).

In [ ]:
numeric = df.select_dtypes('number')
corr = numeric.corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=6)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns, fontsize=6)
ax.set_title('Korelasi fitur numerik')
fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

pairs = corr.where(~(corr.abs() >= 0.9)).stack()
corr.stack().sort_values(ascending=False).head(10)

## 5. Kesimpulan EDA

- Tidak ada missing value maupun duplikat.
- Target tidak seimbang: `Enrolled` hanya ±18% dan paling sulit diprediksi.
- Fitur capaian kurikulum (SKS lulus, nilai) adalah pembeda terkuat antar kelas.
- Banyak fitur kategorikal berkode nominal → gunakan One-Hot Encoding.
- Kelas `Enrolled` perlu perhatian khusus (class weight / metrik macro-F1).